# H121 QC variant comparison (vs legacy baseline)

Compares several H121 QC configurations against each other and against the
legacy BUFR product, using the global GEOS tile/cycle superob caches built by
`build_global_superob_cache.py`. Each cache version applies a different H121
QC config (legacy QC is unchanged across all of them).

**Legacy is used as a comparison baseline, not as ground truth.** It is a
different, older retrieval algorithm with its own (likely larger) error
characteristics. Better or worse agreement with legacy does not directly mean
"more" or "less" accurate — see `projects/ascat_da/report/legacy_vs_h121_qc_flags.md`
Sect. 7 for a worked example of how a stricter QC threshold can *mechanically*
improve agreement with legacy by removing the population where both products
already disagree most (e.g. tropical rainforest), without that meaning the
retrievals themselves got better.

What this notebook focuses on, per QC variant:
1. **Observation counts** — raw obs and tile-cycle superobs, globally and by
   region (tropical rainforest vs rest of globe).
2. **Spatial coverage** — where H121 actually has obs, for a representative day.
3. **SSM value comparison** — bias / RMSE / MAE / correlation against legacy
   on matched (date, platform, tilenum, cycle) tuples, globally and by region.


In [ ]:
import sys, os
from pathlib import Path
from datetime import datetime, timedelta


def _find_lib_root():
    cwd = Path(os.path.abspath(''))
    for p in [cwd] + list(cwd.parents):
        if (p / 'lib').exists() and (p / 'lib' / 'readers.py').exists():
            return p
        for child in p.glob('projects/*/lib'):
            if (child / 'readers.py').exists():
                return child.parent
    raise RuntimeError(f'Cannot find ascat_da/lib/ from {cwd}')


_root = str(_find_lib_root())
if _root not in sys.path:
    sys.path.insert(0, _root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from IPython.display import display

pd.set_option('display.width', 120)


## 1. Configuration

Edit `VERSIONS` to add/remove QC variants — anything with a matching cache under `.cache/global_superobs/` will be picked up automatically; missing ones are skipped with a warning.

In [ ]:
DATES = [datetime(2020, 6, 1) + timedelta(days=i) for i in range(10)]
PLATFORMS = ['Metop-A', 'Metop-B', 'Metop-C']
MAP_DATE = datetime(2020, 6, 5)  # representative day for coverage maps

CACHE_DIR = Path(_root) / '.cache' / 'global_superobs'

# QC variants to compare. Each is a global superob cache built by
# build_global_superob_cache.py with a different H121 QC config (see
# projects/ascat_da/report/legacy_vs_h121_qc_flags.md for the rationale
# behind each one). Legacy QC is identical across all of these.
VERSIONS = {
    'v2 old (subsfc<10, no sens screen)': 'geos_cycle_global_v2_geos_hsaf_qc',
    'v3 default (sens>1dB, subsfc<5)':    'geos_cycle_global_v3_sens1_subsfc5',
    'v4 sens>2dB (subsfc<5)':             'geos_cycle_global_v4_sens2_subsfc5',
    'v5 +bsflag noise_out_of_limits':     'geos_cycle_global_v5_bsflag_noise',
    'v6 +bsflag degraded|noise':          'geos_cycle_global_v6_bsflag_degnoise',
}

# Tropical rainforest bounding boxes (lat0, lon0, lat1, lon1), matching the
# regions checked manually earlier in this investigation.
REGION_BOXES = {
    'Amazon':    (-10, -75, 5, -50),
    'Congo':     (-5, 10, 5, 30),
    'Indonesia': (-10, 95, 5, 140),
}
REGION_ORDER = ['Amazon', 'Congo', 'Indonesia', 'Rest of globe']

print(f"Comparing {len(VERSIONS)} QC variants over {DATES[0].date()} to {DATES[-1].date()}")
for label, tag in VERSIONS.items():
    print(f"  {label:38s} -> {tag}")


## 2. Load caches

Each cache pickle stores one row per (date, product, platform, tilenum, cycle) superob, with the mean lat/lon/SSM of the contributing raw observations. We re-aggregate across split day-boundary cycles the same way the original `legacy_vs_h121_obs.ipynb` does, then tag each row with a region and a QC-variant label.

In [ ]:
def normalize_global_superobs(df):
    """Map source-file dates to GEOS analysis dates and collapse split cycles."""
    out = df.copy()
    source_date = pd.to_datetime(out['date'])
    cycle = out['cycle'].astype('int16')
    out['date'] = (source_date + pd.to_timedelta((cycle // 8).astype(int), unit='D')).dt.strftime('%Y-%m-%d')
    out['cycle'] = (cycle % 8).astype('int16')

    n_obs = out['n_obs'].astype(float)
    out['_ssm_sum'] = out['ssm_pct'] * n_obs
    out['_lat_sum'] = out['lat'] * n_obs
    out['_lon_sum'] = out['lon'] * n_obs

    grouped = (
        out.groupby(['date', 'product', 'platform', 'tilenum', 'cycle'], as_index=False)
        .agg(
            n_obs=('n_obs', 'sum'),
            _ssm_sum=('_ssm_sum', 'sum'),
            _lat_sum=('_lat_sum', 'sum'),
            _lon_sum=('_lon_sum', 'sum'),
        )
    )
    n = grouped['n_obs'].astype(float)
    grouped['ssm_pct'] = grouped['_ssm_sum'] / n
    grouped['lat'] = grouped['_lat_sum'] / n
    grouped['lon'] = grouped['_lon_sum'] / n
    return grouped.drop(columns=['_ssm_sum', '_lat_sum', '_lon_sum'])


def classify_region(lat, lon):
    lat = np.asarray(lat, float)
    lon = np.asarray(lon, float)
    region = np.full(lat.shape, 'Rest of globe', dtype=object)
    for name, (lat0, lon0, lat1, lon1) in REGION_BOXES.items():
        in_box = (lat > lat0) & (lat < lat1) & (lon > lon0) & (lon < lon1)
        region = np.where(in_box, name, region)
    return region


def load_version(label, version):
    frames = []
    missing = []
    for d in DATES:
        f = CACHE_DIR / f'ascat_global_superobs_{d:%Y%m%d}_{version}.pkl'
        if f.exists():
            frames.append(pd.read_pickle(f))
        else:
            missing.append(f.name)
    if not frames:
        print(f"  [skip] {label}: no cache files found for version '{version}'")
        return None
    if missing:
        print(f"  [warn] {label}: missing {len(missing)} of {len(DATES)} daily files "
              f"(still building, or never built?)")
    df = normalize_global_superobs(pd.concat(frames, ignore_index=True))
    df['region'] = classify_region(df['lat'], df['lon'])
    df['qc_version'] = label
    return df


print("Loading cached superobs for each QC variant...")
version_frames = {}
for label, tag in VERSIONS.items():
    df = load_version(label, tag)
    if df is not None:
        version_frames[label] = df
        n_h121 = int((df['product'] == 'h121').sum())
        n_legacy = int((df['product'] == 'legacy').sum())
        print(f"  [ok]   {label}: {n_h121:,} H121 + {n_legacy:,} legacy tile-cycle rows")

if not version_frames:
    raise RuntimeError("No QC variant caches found — build them with build_global_superob_cache.py first.")

all_versions = pd.concat(version_frames.values(), ignore_index=True)
version_order = [v for v in VERSIONS if v in version_frames]


## 3. Observation counts

### 3a. Global: raw obs contributing to superobs, tile-cycle superobs, and unique tiles covered

In [ ]:
counts = (
    all_versions.groupby(['qc_version', 'product'], as_index=False)
    .agg(n_obs_total=('n_obs', 'sum'), n_tile_cycles=('n_obs', 'size'), n_unique_tiles=('tilenum', 'nunique'))
)
counts_h121 = counts[counts['product'] == 'h121'].set_index('qc_version')[
    ['n_obs_total', 'n_tile_cycles', 'n_unique_tiles']]
counts_legacy = counts[counts['product'] == 'legacy'].set_index('qc_version')[
    ['n_obs_total', 'n_tile_cycles', 'n_unique_tiles']]

print("H121 — raw obs contributing to superobs / tile-cycle superobs / unique tiles covered (10-day window):")
display(counts_h121.reindex(version_order))

print()
print("Legacy — shown for reference (QC unchanged across versions; small differences, if any, "
      "come from which days/cycles each cache happened to be built for):")
display(counts_legacy.reindex(version_order))


### 3b. By region (tropical rainforest vs rest of globe)

In [ ]:
regional_counts = (
    all_versions[all_versions['product'] == 'h121']
    .groupby(['qc_version', 'region'], as_index=False)
    .agg(n_obs_total=('n_obs', 'sum'), n_tile_cycles=('n_obs', 'size'), n_unique_tiles=('tilenum', 'nunique'))
)
pivot_obs = regional_counts.pivot(index='qc_version', columns='region', values='n_obs_total').reindex(
    version_order)[REGION_ORDER]
pivot_tiles = regional_counts.pivot(index='qc_version', columns='region', values='n_unique_tiles').reindex(
    version_order)[REGION_ORDER]

print("H121 raw obs contributing to superobs, by region:")
display(pivot_obs)

print()
print("H121 unique tiles covered at least once in the 10-day window, by region:")
display(pivot_tiles)

baseline_label = next((l for l in version_order if l.startswith('v2')), version_order[0])
pct_of_baseline = 100 * pivot_obs.div(pivot_obs.loc[baseline_label])
print()
print(f"As % of '{baseline_label}' obs count (100% = no change from that baseline):")
display(pct_of_baseline.round(1))


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(version_order))
width = 0.8 / len(REGION_ORDER)
for i, region in enumerate(REGION_ORDER):
    vals = pivot_obs[region].reindex(version_order).to_numpy(float)
    ax.bar(x + (i - (len(REGION_ORDER) - 1) / 2) * width, vals, width, label=region)
ax.set_xticks(x)
ax.set_xticklabels(version_order, rotation=20, ha='right')
ax.set_ylabel('H121 raw obs contributing to superobs (log scale)')
ax.set_yscale('log')
ax.set_title('H121 observation counts by region and QC variant')
ax.legend()
plt.tight_layout()
plt.show()


## 4. Spatial coverage

Scatter of unique H121 tiles with at least one accepted observation on
`MAP_DATE`, one panel per QC variant. Rainforest boxes outlined in red for
visual reference.

In [ ]:
map_df = all_versions[(all_versions['product'] == 'h121') &
                       (all_versions['date'] == MAP_DATE.strftime('%Y-%m-%d'))]

n_versions = len(version_order)
ncols = min(3, n_versions)
nrows = int(np.ceil(n_versions / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.2 * nrows),
                          subplot_kw={'projection': ccrs.PlateCarree()})
axes = np.atleast_1d(axes).ravel()

for ax, label in zip(axes, version_order):
    sub = map_df[map_df['qc_version'] == label].drop_duplicates('tilenum')
    ax.set_global()
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4)
    ax.scatter(sub['lon'], sub['lat'], s=1, color='tab:blue', transform=ccrs.PlateCarree(), alpha=0.5)
    for name, (lat0, lon0, lat1, lon1) in REGION_BOXES.items():
        rect = mpatches.Rectangle((lon0, lat0), lon1 - lon0, lat1 - lat0,
                                   fill=False, edgecolor='red', linewidth=1.2,
                                   transform=ccrs.PlateCarree())
        ax.add_patch(rect)
    ax.set_title(f'{label}\n{len(sub):,} tiles with obs on {MAP_DATE.date()}', fontsize=9)

for ax in axes[len(version_order):]:
    ax.axis('off')

plt.tight_layout()
plt.show()


## 5. SSM value comparison vs legacy

Matched on `(date, platform, tilenum, cycle)`, within each QC variant's own
cache (legacy rows are identical across variants, so this is equivalent to
matching against a single shared legacy baseline). `diff_pct = H121 - Legacy`.

**Remember the caveat from the top of this notebook**: legacy is not ground
truth, and a QC variant that removes more of the rainforest population (see
Sect. 3b) will tend to show *better* agreement here for largely mechanical
reasons, not necessarily because its surviving retrievals are more accurate.

In [ ]:
def matched_stats(df_version):
    legacy = df_version[df_version['product'] == 'legacy'][
        ['date', 'platform', 'tilenum', 'cycle', 'ssm_pct']
    ].rename(columns={'ssm_pct': 'legacy_ssm_pct'})
    h121 = df_version[df_version['product'] == 'h121'][
        ['date', 'platform', 'tilenum', 'cycle', 'ssm_pct', 'region', 'n_obs']
    ].rename(columns={'ssm_pct': 'h121_ssm_pct', 'n_obs': 'h121_n_obs'})
    m = legacy.merge(h121, on=['date', 'platform', 'tilenum', 'cycle'], how='inner')
    m['diff_pct'] = m['h121_ssm_pct'] - m['legacy_ssm_pct']
    return m


rows = []
for label in version_order:
    m = matched_stats(version_frames[label])
    for region in ['All'] + REGION_ORDER:
        g = m if region == 'All' else m[m['region'] == region]
        d = g['diff_pct'].to_numpy(float)
        if len(d) == 0:
            continue
        rows.append({
            'qc_version': label,
            'region': region,
            'matched_tile_cycles': len(g),
            'bias_pct': np.mean(d),
            'rmse_pct': np.sqrt(np.mean(d ** 2)),
            'mae_pct': np.mean(np.abs(d)),
            'r': np.corrcoef(g['legacy_ssm_pct'], g['h121_ssm_pct'])[0, 1] if len(g) > 1 else np.nan,
            'h121_n_obs_median': g['h121_n_obs'].median(),
        })

ssm_compare = pd.DataFrame(rows).round(3)

print("--- All (global) ---")
display(ssm_compare[ssm_compare['region'] == 'All'].set_index('qc_version').reindex(version_order))


In [ ]:
for region in REGION_ORDER:
    sub = ssm_compare[ssm_compare['region'] == region].set_index('qc_version').reindex(version_order)
    print(f"--- {region} ---")
    display(sub)
    print()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics = [('bias_pct', 'Bias (H121 - Legacy), % saturation'),
           ('rmse_pct', 'RMSE, % saturation'),
           ('r', 'Correlation (r)')]
for ax, (metric, title) in zip(axes, metrics):
    for region in ['All'] + REGION_ORDER:
        sub = ssm_compare[ssm_compare['region'] == region].set_index('qc_version').reindex(version_order)
        ax.plot(version_order, sub[metric], marker='o', label=region)
    ax.set_title(title, fontsize=10)
    ax.tick_params(axis='x', rotation=25)
    if metric == 'bias_pct':
        ax.axhline(0, color='gray', lw=0.7)
axes[0].legend(fontsize=8, loc='best')
plt.tight_layout()
plt.show()


## 6. Notes

- This notebook compares QC variants against each other and against legacy
  as a common (non-ground-truth) reference. It does not establish which
  variant is "more accurate" — see `legacy_vs_h121_qc_flags.md` for why a
  variant that removes the hardest-to-retrieve regions (rainforest) can look
  better here for reasons unrelated to retrieval accuracy.
- The actual independent-truth validation for the H121 QC thresholds used in
  `v3` (the current default) is Hahn et al. (2026), Sect. 4.2-4.3
  (ISMN/GLDAS/ESA CCI comparison), not this notebook.
- If a variant's panel is missing from Sect. 3/4/5, its cache hasn't finished
  building yet — rerun `build_global_superob_cache.py` for that version and
  re-execute this notebook.
